# Supportive Tables

***
**This file will create supportive tables to be used in the final aggregated table. They will be saved as processed csv files to be imported.**
***

## Supportive Tables

This file will go through each supportive table, use the cleaning function described in the previous file, and perform feature engineering and transform them into useful data for the final database. 

In [22]:
# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

***
***
### Location Lookup Table

The final layout of this file should be as follows:  
LSOA Code | LSOA Name | LAD Code | LAD Name | PFA Code | PFA Name  
  
This will allow for conversions between LSOA's to PFA's and include their LAD's.  
  
From the previous file, we have these functions:

In [23]:
def Clean_LsoaLad(raw_lsoa_lad, dropped_rows):
    lsoa_lad = raw_lsoa_lad[['LSOA21CD', 'LSOA21NM', 'LAD24CD', 'LAD24NM']]

    lsoa_lad = lsoa_lad.rename(columns={
        'LSOA21CD': 'lsoa_code',
        'LSOA21NM': 'lsoa_name',
        'LAD24CD': 'lad_code',
        'LAD24NM': 'lad_name'
    })

    return lsoa_lad, dropped_rows

def Clean_LadPfa(raw_lad_pfa, dropped_rows):
    lad_pfa = raw_lad_pfa[['LAD25CD', 'LAD25NM', 'PFA25CD', 'PFA25NM']]

    lad_pfa = lad_pfa.rename(columns={
        'LAD25CD': 'lad_code',
        'LAD25NM': 'lad_name',
        'PFA25CD': 'pfa_code',
        'PFA25NM': 'pfa_name'
    })

    dropped_rows['duplicates'] = lad_pfa.duplicated().sum()

    lad_pfa = lad_pfa.drop_duplicates()

    return lad_pfa, dropped_rows

***
#### Ingestion

In [24]:
# Import LSOA <-> LAD data:
raw_lsoa_lad = pd.read_csv('../Data/Raw/lookup/lsoa-lad.csv')

raw_lsoa_lad.head(5)

,LSOA21CD,LSOA21NM,LSOA21NMW,WD24CD,WD24NM,WD24NMW,LAD24CD,LAD24NM,LAD24NMW,ObjectId
0,E01012000,Hartlepool 007E,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,1
1,E01011964,Hartlepool 007B,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,2
2,E01011999,Hartlepool 007D,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,3
3,E01011967,Hartlepool 007C,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,4
4,E01011951,Hartlepool 007A,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,5


In [25]:
# Import LAD <-> PFA raw data
raw_lad_pfa = pd.read_csv('../Data/Raw/lookup/lad-pfa.csv')

raw_lad_pfa.head(5)

,LAD25CD,LAD25NM,CSP25CD,CSP25NM,PFA25CD,PFA25NM,ObjectId
0,E06000058,"Bournemouth, Christchurch and Poole",E22000367,Dorset,E23000039,Dorset,1
1,E06000059,Dorset,E22000367,Dorset,E23000039,Dorset,2
2,E06000060,Buckinghamshire,E22000303,Aylesbury Vale,E23000029,Thames Valley,3
3,E06000060,Buckinghamshire,E22000306,Chiltern,E23000029,Thames Valley,4
4,E06000060,Buckinghamshire,E22000311,South Bucks,E23000029,Thames Valley,5


***
#### Cleaning and Validation

In [26]:
dropped_rows = {}

In [27]:
clean_lsoa_lad, dropped_rows = Clean_LsoaLad(raw_lsoa_lad, dropped_rows)

clean_lsoa_lad.head()

,lsoa_code,lsoa_name,lad_code,lad_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool
1,E01011964,Hartlepool 007B,E06000001,Hartlepool
2,E01011999,Hartlepool 007D,E06000001,Hartlepool
3,E01011967,Hartlepool 007C,E06000001,Hartlepool
4,E01011951,Hartlepool 007A,E06000001,Hartlepool


In [28]:
for reason in dropped_rows:
    print(f'{reason}: {dropped_rows[reason]}')

In [29]:
clean_lad_pfa, dropped_rows = Clean_LadPfa(raw_lad_pfa, dropped_rows)

clean_lsoa_lad.head()

,lsoa_code,lsoa_name,lad_code,lad_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool
1,E01011964,Hartlepool 007B,E06000001,Hartlepool
2,E01011999,Hartlepool 007D,E06000001,Hartlepool
3,E01011967,Hartlepool 007C,E06000001,Hartlepool
4,E01011951,Hartlepool 007A,E06000001,Hartlepool


In [30]:
for reason in dropped_rows:
    print(f'{reason}: {dropped_rows[reason]}')

duplicates: 14


***
#### Feature Engineering & Transformation

This section will create a final lookup table to be used in the rest of the data pipeline to give locations different grain.

***
*Small Explanation of Barnsley and Sheffield Data:*  
Before 2025, Barnsley and Sheffield had LAD codes of E08000016 and E08000019 respectively.  
In 2025, These were updated due to voting areas to E08000038 and E08000039 respectively.  
By using purely the location lookup table, issues regarding these LAD's can be solved by fixing them here.  
By using only the modern locations, there may be some older data which shows in the wrong location - however this will only account for a minor amount, and is therefore a sound assumption.
  
The data comes from these links:  
Barnsley: __https://www.ons.gov.uk/explore-local-statistics/areas/E08000038-barnsley__  
Sheffield: __https://www.ons.gov.uk/explore-local-statistics/areas/E08000039-sheffield__
***

In [31]:
# Change lsoa->lad dataset outdated data

# Update old Sheffield LAD code to new Sheffield LAD code
clean_lsoa_lad.loc[clean_lsoa_lad['lad_code'] == 'E08000016', 'lad_code'] = 'E08000038'

# Update old Barnsley LAD code to new Barnsley LAD code
clean_lsoa_lad.loc[clean_lsoa_lad['lad_code'] == 'E08000019', 'lad_code'] = 'E08000039'

clean_lsoa_lad[clean_lsoa_lad['lad_code'].isin(['E08000038', 'E08000039'])]

,lsoa_code,lsoa_name,lad_code,lad_name
24051,E01007429,Barnsley 024C,E08000038,Barnsley
24054,E01007444,Barnsley 012F,E08000038,Barnsley
24058,E01007428,Barnsley 024B,E08000038,Barnsley
24059,E01007334,Barnsley 009A,E08000038,Barnsley
24061,E01007382,Barnsley 019A,E08000038,Barnsley
...,...,...,...,...
25184,E01008134,Sheffield 005B,E08000039,Sheffield
25187,E01007888,Sheffield 003A,E08000039,Sheffield
25190,E01007899,Sheffield 003E,E08000039,Sheffield
25193,E01007901,Sheffield 003G,E08000039,Sheffield


In [32]:
# Merge tables on lad code

lookup_table = pd.merge(clean_lsoa_lad, clean_lad_pfa, how='left', on=['lad_code', 'lad_name'])

lookup_table.head(5)

,lsoa_code,lsoa_name,lad_code,lad_name,pfa_code,pfa_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool,E23000013,Cleveland
1,E01011964,Hartlepool 007B,E06000001,Hartlepool,E23000013,Cleveland
2,E01011999,Hartlepool 007D,E06000001,Hartlepool,E23000013,Cleveland
3,E01011967,Hartlepool 007C,E06000001,Hartlepool,E23000013,Cleveland
4,E01011951,Hartlepool 007A,E06000001,Hartlepool,E23000013,Cleveland


Finally, check cleanliness of the table:

In [33]:
lookup_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lsoa_code  35672 non-null  object
 1   lsoa_name  35672 non-null  object
 2   lad_code   35672 non-null  object
 3   lad_name   35672 non-null  object
 4   pfa_code   35672 non-null  object
 5   pfa_name   35672 non-null  object
dtypes: object(6)
memory usage: 1.6+ MB


Correct data types.

In [34]:
lookup_table.isnull().sum()

lsoa_code    0
lsoa_name    0
lad_code     0
lad_name     0
pfa_code     0
pfa_name     0
dtype: int64

No null values.

In [35]:
lookup_table.duplicated().sum()

np.int64(0)

#### Exporting the Table

In [36]:
## Export the code to processed folder as a csv
lookup_table.to_csv('../Data/Processed/location-lookup-table.csv', index=False)

print(f'location-lookup-table.csv File successfully created: {Path('../Data/Processed/location-lookup-table.csv').exists()}')

location-lookup-table.csv File successfully created: True


***
***
### Population

The final layout of this file should be as follows:  
LSOA Code | Year | Population  
  
This will allow for the final database to know the population of the location where the crime took place.

From previous, the cleaning function:

In [37]:
def Clean_Population(raw_population, year):
    population = raw_population[['LSOA 2021 Code', 'Total']]

    population = population.rename(columns={
        'LSOA 2021 Code': 'lsoa_code',
        'Total': 'population',
    })

    return population    

#### Ingestion

<div class="alert alert-block alert-warning">
<b>Warning:</b> This section may take some time to import, there are 35,000 rows per year. It usually takes ~1min 30seconds to complete each import.
</div>

In [38]:
# Import Population Data Raw
raw_pop_2022 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2022 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2022.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1876
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1117
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1260
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1635
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,1984


In [39]:
raw_pop_2023 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2023 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2023.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1925
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1177
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1320
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1670
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2075


In [40]:
raw_pop_2024 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2024 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2024.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1898
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1247
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1393
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1669
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,2303


#### Cleaning and Validation

In [41]:
clean_pop_2022 = Clean_Population(raw_pop_2022, 2022)

clean_pop_2022.head()

,lsoa_code,population
0,E01011949,1876
1,E01011950,1117
2,E01011951,1260
3,E01011952,1635
4,E01011953,1984


In [42]:
clean_pop_2023 = Clean_Population(raw_pop_2023, 2023)

clean_pop_2023.head()

,lsoa_code,population
0,E01011949,1925
1,E01011950,1177
2,E01011951,1320
3,E01011952,1670
4,E01011953,2075


In [43]:
clean_pop_2024 = Clean_Population(raw_pop_2024, 2024)

clean_pop_2024.head()

,lsoa_code,population
0,E01011949,1898
1,E01011950,1247
2,E01011951,1393
3,E01011952,1669
4,E01011953,2303


***
In order to create estimates for 2025 and 2026, we will; merge these 3 tables, create estimates, create new databases from those estimates.

In [44]:
pop_growth = clean_pop_2022.merge(
    clean_pop_2023,
    on='lsoa_code',
    how='inner',
    suffixes=('_2022', '_2023')
).merge(
    clean_pop_2024,
    on='lsoa_code',
    how='inner'
)

pop_growth = pop_growth.rename(columns={
    'population': 'population_2024'
})

pop_growth.head()

,lsoa_code,population_2022,population_2023,population_2024
0,E01011949,1876,1925,1898
1,E01011950,1117,1177,1247
2,E01011951,1260,1320,1393
3,E01011952,1635,1670,1669
4,E01011953,1984,2075,2303


In [45]:
# Calculate average annual change
pop_growth['change_2022_2023'] = (
    pop_growth['population_2023'] - pop_growth['population_2022']
)

pop_growth['change_2023_2024'] = (
    pop_growth['population_2024'] - pop_growth['population_2023']
)

pop_growth['avg_annual_change'] = (
    pop_growth[['change_2022_2023', 'change_2023_2024']].mean(axis=1)
)

pop_growth.head()

,lsoa_code,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change
0,E01011949,1876,1925,1898,49,-27,11.0
1,E01011950,1117,1177,1247,60,70,65.0
2,E01011951,1260,1320,1393,60,73,66.5
3,E01011952,1635,1670,1669,35,-1,17.0
4,E01011953,1984,2075,2303,91,228,159.5


In [46]:
# Add new column for 2025 estimated population
# ASSUMPTION: Growth rate will stay, on average, the same for the next 2 years after the data

pop_growth['population_2025'] = pop_growth['population_2024'] + pop_growth['avg_annual_change']

pop_growth.head()

,lsoa_code,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025
0,E01011949,1876,1925,1898,49,-27,11.0,1909.0
1,E01011950,1117,1177,1247,60,70,65.0,1312.0
2,E01011951,1260,1320,1393,60,73,66.5,1459.5
3,E01011952,1635,1670,1669,35,-1,17.0,1686.0
4,E01011953,1984,2075,2303,91,228,159.5,2462.5


In [47]:
# Add a new columns for 2026 estimated population
pop_growth['population_2026'] = pop_growth['population_2025'] + pop_growth['avg_annual_change']

pop_growth.head()

,lsoa_code,population_2022,population_2023,population_2024,change_2022_2023,change_2023_2024,avg_annual_change,population_2025,population_2026
0,E01011949,1876,1925,1898,49,-27,11.0,1909.0,1920.0
1,E01011950,1117,1177,1247,60,70,65.0,1312.0,1377.0
2,E01011951,1260,1320,1393,60,73,66.5,1459.5,1526.0
3,E01011952,1635,1670,1669,35,-1,17.0,1686.0,1703.0
4,E01011953,1984,2075,2303,91,228,159.5,2462.5,2622.0


In [48]:
clean_pop_2025 = pop_growth[['lsoa_code', 'population_2025']]

clean_pop_2025 = clean_pop_2025.rename(columns={
    'population_2025': 'population'
})

clean_pop_2025.head()

,lsoa_code,population
0,E01011949,1909.0
1,E01011950,1312.0
2,E01011951,1459.5
3,E01011952,1686.0
4,E01011953,2462.5


In [49]:
clean_pop_2026 = pop_growth[['lsoa_code', 'population_2026']]

clean_pop_2026 = clean_pop_2026.rename(columns={
    'population_2026': 'population'
})

clean_pop_2026.head()

,lsoa_code,population
0,E01011949,1920.0
1,E01011950,1377.0
2,E01011951,1526.0
3,E01011952,1703.0
4,E01011953,2622.0


Now we have all the databases required, we will add the year column to each, and concatinate them.

In [50]:
def AddYear(population_data, year):
    population_data['year'] = year
    return population_data

clean_pop_2022 = AddYear(clean_pop_2022, 2022)
clean_pop_2023 = AddYear(clean_pop_2023, 2023)
clean_pop_2024 = AddYear(clean_pop_2024, 2024)
clean_pop_2025 = AddYear(clean_pop_2025, 2025)
clean_pop_2026 = AddYear(clean_pop_2026, 2026)

In [51]:
population_final = pd.concat([clean_pop_2022, clean_pop_2023, clean_pop_2024, clean_pop_2025, clean_pop_2026], ignore_index=True)

population_final.sample(10)

,lsoa_code,population,year
169730,E01010387,1581.0,2026
72734,E01013317,1379.0,2024
129582,E01005854,1712.5,2025
114127,E01017471,1974.5,2025
141716,W01001037,1875.5,2025
8916,E01019221,1780.0,2022
168420,E01009122,1685.0,2026
146467,E01015772,1547.0,2026
56169,E01034830,1285.0,2023
113819,E01031857,1487.0,2025


Final check for cleanliness

In [52]:
population_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178360 entries, 0 to 178359
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   lsoa_code   178360 non-null  object 
 1   population  178360 non-null  float64
 2   year        178360 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 4.1+ MB


In [53]:
population_final.isnull().sum()

lsoa_code     0
population    0
year          0
dtype: int64

In [54]:
population_final.duplicated().sum()

np.int64(0)

#### Exporting

In [55]:
## Export the code to processed folder as a csv
population_final.to_csv('../Data/Processed/population.csv', index=False)

print(f'lookup-table.csv File successfully created: {Path('../Data/Processed/population.csv').exists()}')

lookup-table.csv File successfully created: True


***
***
### Deprivation

The final layout of this file should have the structure of:  
LSOA Code | imd_score | incm_score | empl_score | edcn_score | hous_score

Using the previous file to define a cleaning process:

In [56]:
def Clean_Deprivation(raw_depr):
    # Get useful columns
    depr = raw_depr[[
        'LSOA code (2021)', 
        'Index of Multiple Deprivation (IMD) Score',
        'Income Score (rate)',
        'Employment Score (rate)',
        'Education, Skills and Training Score',
        'Barriers to Housing and Services Score'
    ]]

    # Rename columns
    depr = depr.rename(columns={
        'LSOA code (2021)': 'lsoa_code',
        'Index of Multiple Deprivation (IMD) Score': 'imd_score',
        'Income Score (rate)': 'incm_score',
        'Employment Score (rate)': 'empl_score',
        'Education, Skills and Training Score': 'edcn_score',
        'Barriers to Housing and Services Score': 'hous_score'
    })

    return depr

#### Ingestion

In [57]:
# Import Deprivation Data Raw
raw_depr = pd.read_csv(f'../Data/Raw/deprivation/deprivation.csv')

raw_depr.head()

,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2022,Dependent Children aged 0-15: mid 2022,Older population aged 60 and over: mid 2022,Working age population 18-66 (for use with Employment Deprivation Domain): mid 2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,...,1.207,1105,1,1.414,1586,1,1795,149,520,1248
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,...,0.355,9591,3,1.839,592,1,1671,81,387,1324
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8,...,0.318,10175,4,1.679,903,1,1896,136,432,1469
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,...,0.012,15502,5,2.065,303,1,1737,177,160,1448
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,...,0.399,8934,3,0.400,9136,3,1837,397,225,1260


#### Cleaning and Validation

In [58]:
clean_depr = Clean_Deprivation(raw_depr)

clean_depr.head()

,lsoa_code,imd_score,incm_score,empl_score,edcn_score,hous_score
0,E01000001,8.742,0.013,0.014,0.004,10.950
1,E01000002,4.722,0.018,0.010,0.169,6.703
2,E01000003,9.250,0.107,0.064,3.269,9.735
3,E01000005,19.884,0.211,0.104,17.852,24.623
4,E01000006,25.307,0.343,0.120,25.442,38.025


In [59]:
clean_depr['lsoa_code'].duplicated().sum()

np.int64(0)

No need for any transformations to be done, this is already in the end form.

#### Exporting

In [60]:
## Export the code to processed folder as a csv
clean_depr.to_csv('../Data/Processed/deprivation.csv', index=False)

print(f'deprivation.csv File successfully created: {Path('../Data/Processed/deprivation.csv').exists()}')

deprivation.csv File successfully created: True


***
***
### Crime Severity

The final layout of this dataframe will have the structure:  
Crime Category | Average Weighting   
  
Using the previous file, the cleaning function is defined as:

In [61]:
def Clean_Severity(raw_severity):
    # Change weight to integer
    raw_severity['Weight'] = raw_severity['Weight'].astype(str).str.replace(',', '', regex=False)
    raw_severity['Weight'] = pd.to_numeric(raw_severity['Weight'], errors='coerce')

    # Set null values of Crime Category to 'No Category'
    raw_severity['Crime Category'] = raw_severity['Crime Category'].fillna('No Category')

    # Set null values of Crime Index to 'No Index'
    raw_severity['Crime Index'] = raw_severity['Crime Index'].fillna('No Index')

    # Rename columns
    raw_severity = raw_severity.rename(columns={
        'Crime Category': 'crime_cat',
        'Weight': 'weight'
    })

    return raw_severity

#### Ingestion

In [62]:
# Import crime severity categorised data set
raw_sev = pd.read_csv(f'../Data/Processed/crime-severity-raw/crime-severity-categorised.csv')

raw_sev.head()

,Crime Index,Offence,Weight,Crime Category
0,"1, 4.1/10/2",Homicide,"7,979",Violence and sexual offences
1,2,Attempted murder,"4,663",Violence and sexual offences
2,4.3,Intentional destruction of viable unborn child,15,Violence and sexual offences
3,4.4,Causing death or serious injury by dangerous d...,"1,092",Violence and sexual offences
4,4.6,Causing death by careless driving when under t...,"1,595",Violence and sexual offences


#### Cleaning and Validation

In [63]:
clean_sev = Clean_Severity(raw_sev)

clean_sev.head()

,Crime Index,Offence,weight,crime_cat
0,"1, 4.1/10/2",Homicide,7979,Violence and sexual offences
1,2,Attempted murder,4663,Violence and sexual offences
2,4.3,Intentional destruction of viable unborn child,15,Violence and sexual offences
3,4.4,Causing death or serious injury by dangerous d...,1092,Violence and sexual offences
4,4.6,Causing death by careless driving when under t...,1595,Violence and sexual offences


#### Feature Engineering and Transform

Firstly, it needs to be decided which average will be used.

In [64]:
## Group Table
sev_by_crime_category = clean_sev.groupby(['crime_cat'])

## Create Columns
num_items = sev_by_crime_category['weight'].count()

mean_weight = sev_by_crime_category['weight'].mean()

median_weight = sev_by_crime_category['weight'].median()

min_weight = sev_by_crime_category['weight'].min()

max_weight = sev_by_crime_category['weight'].max()

std_deviation_weight = sev_by_crime_category['weight'].std()

##Formulate Table
crime_category_severity = pd.DataFrame({
    'num_items': num_items,
    'mean_weight': mean_weight,
    'median_weight': median_weight,
    'min_weight': min_weight,
    'max_weight': max_weight,
    'std_deviation_weight': std_deviation_weight
})

## Visulaise Table
display(crime_category_severity)

,num_items,mean_weight,median_weight,min_weight,max_weight,std_deviation_weight
crime_cat,,,,,,
Bicycle Theft,1,16.000000,16.0,16,16,NaN
Burglary,16,703.250000,438.0,117,2127,697.764765
Criminal damage and arson,12,132.000000,19.0,7,837,255.916890
Drugs,5,105.000000,9.0,3,497,219.157478
No Category,8,280.250000,106.0,106,803,322.648305
Other crime,93,161.978495,86.0,4,4392,454.636285
Other theft,8,143.375000,51.5,7,803,268.795694
Possession of weapons,7,367.142857,75.0,55,1365,490.724101
Public order,9,401.000000,365.0,10,1880,575.706088


***
Looking at these stats, taking the median seems to give a better value, as the skew from large and small data is much less.  
Furthermore, by taking the average - given we are working with large datasets - the skew will become more obvious. This is because lower weighted crimes will be committed more often.  
With median: The more common crimes will be weighted slightly higher than they should, the more dangerous crimes will be rated much lower than they should.  
With mean: The more common crimes will be rated much higher than they should, the more dangerous crimes will be rated lower than they should.  
  
**Assumption:** Median is the best average to use for crime severity weighting.
***

In [65]:
sev_lookup = (
    clean_sev
    .groupby('crime_cat', as_index=False)
    .agg(avg_weight=('weight', 'median'))
)

sev_lookup['crime_cat'] = sev_lookup['crime_cat'].str.lower()

In [66]:
sev_lookup

,crime_cat,avg_weight
0,bicycle theft,16.0
1,burglary,438.0
2,criminal damage and arson,19.0
3,drugs,9.0
4,no category,106.0
5,other crime,86.0
6,other theft,51.5
7,possession of weapons,75.0
8,public order,365.0
9,robbery,746.0


Now every crime (except anti-social behaviour - which we will come to later) has an average crime weighting.  
These are based off of the average sentence for each crime.

#### Exporting

In [67]:
## Output Final Table to csv
sev_lookup.to_csv('../Data/Processed/crime-severity.csv', index=True)

print(f'crime-severity.csv File successfully created: {Path('../Data/Processed/crime-severity.csv').exists()}')

crime-severity.csv File successfully created: True
